In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
RNG=np.random.default_rng(20260726)
B=2000   # bootstrap resamples
print('ready:', os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [7]:
# =============================================================================
# C1 (TIER 1) - CALIBRATION-SET SIZE AUDIT.
# The paper claims the TSC vs SHC contrast "holds the calibration-set size fixed".
# The feasibility table shows NSL R2L with 149 SHC source points vs 299 TSC points.
# If the matched draws do NOT equalise n_cal, part of the headline gap is a
# sample-size effect on the quantile, not a source-vs-target effect.
# n_cal is recorded per coverage cell, so this is a direct check.
# =============================================================================
LONG=config.PROC_DIR/'coverage_long_nslkdd.parquet'
assert LONG.exists(), f'missing {LONG}'
nl=pd.read_parquet(LONG)
print('long parquet:', nl.shape, '| cols:', list(nl.columns))
assert 'n_cal' in nl.columns, 'n_cal not recorded in long parquet; cannot audit calibration size here'
prim=nl[(nl['score']=='aps')&(nl['variant']=='mondrian')&(np.isclose(nl['alpha'],0.05))]

print('\nn_cal by protocol x class (primary spec, all rungs):')
tab=prim.groupby(['class','protocol'])['n_cal'].agg(['mean','min','max']).round(1)
print(tab.to_string())

print('\nFOCAL R2L n_cal, SHC vs TSC:')
r=prim[prim['class']=='R2L'].groupby('protocol')['n_cal'].agg(['mean','min','max','count'])
print(r.round(1).to_string())
shc_n=float(r.loc['SHC','mean']); tsc_n=float(r.loc['TSC','mean'])
ratio=shc_n/tsc_n if tsc_n else np.nan
EQUALISED = abs(ratio-1.0) < 0.05
print(f'\nSHC mean n_cal={shc_n:.1f}  TSC mean n_cal={tsc_n:.1f}  ratio={ratio:.3f}')
print('VERDICT:', 'EQUALISED - size confound ruled out' if EQUALISED else
      'NOT EQUALISED - the TSC/SHC gap is confounded with calibration-set size; must be disclosed and, if possible, re-run size-matched')
c1={'focal_shc_ncal':round(shc_n,1),'focal_tsc_ncal':round(tsc_n,1),
    'ratio':round(float(ratio),3),'equalised':bool(EQUALISED),
    'per_class':tab.reset_index().to_dict('records')}


long parquet: (864000, 18) | cols: ['class', 'n_eval', 'n_covered', 'n_cal', 'q_hat', 'feasible', 'mean_set_size', 'empty_rate', 'full_rate', 'rung', 'realization', 'arch', 'seed', 'protocol', 'variant', 'score', 'alpha', 'coverage']

n_cal by protocol x class (primary spec, all rungs):
                          mean    min    max
class        protocol                       
DoS          REC         772.1    770    774
             SHC        6889.0   6889   6889
             TSC         772.1    770    774
Normal       REC        1008.0   1008   1008
             SHC       10101.0  10101  10101
             TSC        1008.0   1008   1008
Probe        REC         249.9    248    251
             SHC        1748.0   1748   1748
             TSC         249.9    248    251
R2L          REC         297.1    295    299
             SHC         149.0    149    149
             TSC         297.1    295    299
U2R          REC           5.5      4      7
             SHC           7.0      7

In [8]:
# =============================================================================
# C2 - MARGINAL vs MONDRIAN. The paper's motivating claim is that marginal
# coverage hides per-class failure. The data exists (variant column) and has
# never been reported. This makes the claim a measured result instead of rhetoric.
# =============================================================================
mar=nl[(nl['score']=='aps')&(nl['variant']=='marginal')&(np.isclose(nl['alpha'],0.05))]
mon=nl[(nl['score']=='aps')&(nl['variant']=='mondrian')&(np.isclose(nl['alpha'],0.05))]
print('marginal rows:',len(mar),'| mondrian rows:',len(mon))
# the marginal variant uses ONE shared quantile, so per-class feasibility may be
# recorded as False for rare classes; if that empties the table, drop the filter.
MAR_FEAS = bool(mar['feasible'].any())
if not MAR_FEAS:
    print('NOTE: no marginal rows flagged feasible (per-class rule); using all marginal rows.')

rows=[]
for proto in ['REC','TSC','SHC']:
    # marginal variant: coverage pooled over classes (one guarantee for all)
    mm=mar[(mar['protocol']==proto)&(mar['feasible'])] if MAR_FEAS else mar[mar['protocol']==proto]
    mm_all=float(np.average(mm['coverage'],weights=mm['n_eval'])) if len(mm) else np.nan
    mm_focal=float(np.average(mm[mm['class']=='R2L']['coverage'],
                   weights=mm[mm['class']=='R2L']['n_eval'])) if len(mm[mm['class']=='R2L']) else np.nan
    dd=mon[(mon['protocol']==proto)&(mon['feasible'])]
    dd_focal=float(dd[dd['class']=='R2L']['coverage'].mean()) if len(dd[dd['class']=='R2L']) else np.nan
    rows.append({'protocol':proto,'marginal_overall':round(mm_all,4),
                 'marginal_focal_R2L':round(mm_focal,4),'mondrian_focal_R2L':round(dd_focal,4)})
marg_tbl=pd.DataFrame(rows)
print('\nNSL-KDD, alpha=0.05: does the MARGINAL guarantee hide the focal failure?')
print(marg_tbl.to_string(index=False))
print('\nread: if marginal_overall for SHC sits near 0.95 while marginal_focal_R2L (the')
print('      focal class under the SINGLE marginal quantile) is far below, then a')
print('      marginal guarantee is satisfied while the focal class is badly undercovered.')


marginal rows: 54000 | mondrian rows: 54000

NSL-KDD, alpha=0.05: does the MARGINAL guarantee hide the focal failure?
protocol  marginal_overall  marginal_focal_R2L  mondrian_focal_R2L
     REC            0.9507              0.8591              0.9562
     TSC            0.9507              0.8595              0.9536
     SHC            0.7200              0.0714              0.0860

read: if marginal_overall for SHC sits near 0.95 while marginal_focal_R2L (the
      focal class under the SINGLE marginal quantile) is far below, then a
      marginal guarantee is satisfied while the focal class is badly undercovered.


In [9]:
# =============================================================================
# C3 - BOOTSTRAP CIs on every headline number. Cluster by SEED (the model is the
# resampling unit); for NSL also cluster by REALIZATION. No headline number in a
# Q1 paper should be a bare point estimate.
# NOTE: the source partition is drawn ONCE and shared by all models, so these CIs
# capture model+draw variability but NOT partition variability. Stated as a limit.
# =============================================================================
def cluster_boot_mean(df, value_col, cluster_col, B=B, rng=RNG):
    cl=df[cluster_col].unique()
    if len(cl)<2: return (np.nan,np.nan,np.nan)
    means={c:df[df[cluster_col]==c][value_col].mean() for c in cl}
    arr=np.array([means[c] for c in cl])
    boots=np.array([rng.choice(arr,len(arr),replace=True).mean() for _ in range(B)])
    return (float(arr.mean()), float(np.percentile(boots,2.5)), float(np.percentile(boots,97.5)))

ci_rows=[]
# NSL focal across rungs (the dose-response ladder = proposed PRIMARY analysis)
nsl_p=pd.read_csv(config.REPORTS_DIR/'coverage_primary_nslkdd.csv')
for rung in sorted(nsl_p['rung'].unique()):
    s=nsl_p[(nsl_p['class']=='R2L')&(np.isclose(nsl_p['alpha'],0.05))&
            (nsl_p['protocol']=='SHC')&(np.isclose(nsl_p['rung'],rung))]
    m,lo,hi=cluster_boot_mean(s,'coverage','seed')
    ci_rows.append({'dataset':'nslkdd','stratum':f'R2L SHC rung {rung:.2f}','mean':round(m,4),
                    'ci_lo':round(lo,4),'ci_hi':round(hi,4),'n_cells':len(s)})
# CIC + UGR focal at each alpha
for f,cls,lab in [('coverage_primary_cicids2017.csv','DoS','cicids2017'),
                  ('coverage_primary_ugr16.csv','nerisbotnet','ugr16')]:
    d=pd.read_csv(config.REPORTS_DIR/f)
    for a in [0.05,0.10,0.20]:
        s=d[(d['class']==cls)&(np.isclose(d['alpha'],a))&(d['protocol']=='SHC')]
        if not len(s): continue
        m,lo,hi=cluster_boot_mean(s,'coverage','seed')
        ci_rows.append({'dataset':lab,'stratum':f'{cls} SHC alpha {a}','mean':round(m,4),
                        'ci_lo':round(lo,4),'ci_hi':round(hi,4),'n_cells':len(s)})
# UGR discovered scan collapse
ug=pd.read_csv(config.REPORTS_DIR/'coverage_primary_ugr16.csv')
for cls in ['scan11','scan44']:
    s=ug[(ug['class']==cls)&(np.isclose(ug['alpha'],0.05))&(ug['protocol']=='SHC')]
    m,lo,hi=cluster_boot_mean(s,'coverage','seed')
    ci_rows.append({'dataset':'ugr16','stratum':f'{cls} SHC alpha 0.05','mean':round(m,4),
                    'ci_lo':round(lo,4),'ci_hi':round(hi,4),'n_cells':len(s)})
ci_tbl=pd.DataFrame(ci_rows)
print(f'SEED-CLUSTERED 95% BOOTSTRAP CIs (B={B}):')
print(ci_tbl.to_string(index=False))


SEED-CLUSTERED 95% BOOTSTRAP CIs (B=2000):
   dataset                    stratum   mean  ci_lo  ci_hi  n_cells
    nslkdd          R2L SHC rung 0.00 0.1431 0.1284 0.1617      600
    nslkdd          R2L SHC rung 0.20 0.1139 0.1026 0.1286      600
    nslkdd          R2L SHC rung 0.40 0.0848 0.0763 0.0959      600
    nslkdd          R2L SHC rung 0.60 0.0582 0.0522 0.0662      600
    nslkdd          R2L SHC rung 0.80 0.0298 0.0267 0.0339      600
cicids2017         DoS SHC alpha 0.05 0.6039 0.5970 0.6111      150
cicids2017          DoS SHC alpha 0.1 0.5565 0.5499 0.5643      150
cicids2017          DoS SHC alpha 0.2 0.4786 0.4719 0.4869      150
     ugr16 nerisbotnet SHC alpha 0.05 0.9473 0.9469 0.9476       30
     ugr16  nerisbotnet SHC alpha 0.1 0.8953 0.8948 0.8959       30
     ugr16  nerisbotnet SHC alpha 0.2 0.7904 0.7898 0.7911       30
     ugr16      scan11 SHC alpha 0.05 0.5350 0.5247 0.5441       30
     ugr16      scan44 SHC alpha 0.05 0.7990 0.7943 0.8039       30


In [10]:
# =============================================================================
# C4 (TIER 1) - INCLUSION-RULE AUDIT. NSL U2R has 7 source calibration points vs
# a feasibility floor of 19 at alpha=0.05, so it is EXCLUDED from the coverage
# analysis - yet it appears in the mechanism and monitor tables, and the paper
# cites "U2R undercovers by 0.41" as a headline blind-spot example.
# Recompute the mechanism and monitor statistics WITH and WITHOUT infeasible
# cells so the paper can state one rule and show the result is not an artefact.
# =============================================================================
fb=pd.read_csv(config.REPORTS_DIR/'feasibility_binding_nslkdd.csv')
infeas=set(fb[(np.isclose(fb['alpha'],0.05))&(~fb['shc_feasible'])]['class'])
print('NSL classes INFEASIBLE at alpha=0.05 (SHC):', infeas)

me=pd.read_csv(config.REPORTS_DIR/'score_shift_explainability.csv'); me['ds']=me['dataset'].str.split(':').str[0]
mo=pd.read_csv(config.REPORTS_DIR/'monitor_labelfree.csv'); mo['ds']=mo['dataset'].str.split(':').str[0]
def is_excluded(df): return (df['ds']=='nslkdd') & (df['class'].isin(infeas))
me_ex, mo_ex = is_excluded(me), is_excluded(mo)
print(f'mechanism cells dropped if rule applied: {int(me_ex.sum())}/{len(me)}')
print(f'monitor  cells dropped if rule applied: {int(mo_ex.sum())}/{len(mo)}')

def monitor_stats(df):
    r_d=df['drift_labelfree'].rank(pct=True); r_m=df['pred_mass_drop'].clip(lower=0).rank(pct=True)
    det=np.where(df['low_support'].fillna(False),r_m,r_d)
    y=(df['undercoverage']>0.05).astype(int)
    rho,p=stats.spearmanr(det,df['undercoverage'])
    auc=roc_auc_score(y,det) if y.nunique()>1 else np.nan
    return float(rho),float(auc),len(df)
def mech_stats(df):
    rho,p=stats.spearmanr(df['score_KS'],df['undercoverage']); return float(rho),float(p),len(df)

m_all=mech_stats(me); m_fea=mech_stats(me[~me_ex])
o_all=monitor_stats(mo); o_fea=monitor_stats(mo[~mo_ex])
print(f'\nMECHANISM  all cells: rho={m_all[0]:.3f} n={m_all[2]}  |  feasible-only: rho={m_fea[0]:.3f} n={m_fea[2]}')
print(f'MONITOR    all cells: rho={o_all[0]:.3f} AUC={o_all[1]:.3f} n={o_all[2]}  |  feasible-only: rho={o_fea[0]:.3f} AUC={o_fea[1]:.3f} n={o_fea[2]}')
print('\n=> if the feasible-only numbers hold, the paper can adopt ONE inclusion rule')
print('   and re-anchor the blind-spot example on UGR scan11 (feasible, n_cal=7500).')


NSL classes INFEASIBLE at alpha=0.05 (SHC): {'U2R'}
mechanism cells dropped if rule applied: 3/60
monitor  cells dropped if rule applied: 3/60

MECHANISM  all cells: rho=0.925 n=60  |  feasible-only: rho=0.929 n=57
MONITOR    all cells: rho=0.725 AUC=0.895 n=60  |  feasible-only: rho=0.796 AUC=0.924 n=57

=> if the feasible-only numbers hold, the paper can adopt ONE inclusion rule
   and re-anchor the blind-spot example on UGR scan11 (feasible, n_cal=7500).


In [11]:
# =============================================================================
# C5 - HONEST EFFECT SIZES.
# (a) mechanism at CLASS level (architectures on the same dataset:class are not
#     independent observations), with a cluster bootstrap CI on rho.
# (b) monitor per dataset (the pooled AUROC hides that NSL is weak), with a
#     bootstrap CI on AUROC and sensitivity to the undercoverage threshold.
# =============================================================================
# (a) mechanism
cls_lvl=me.groupby(['dataset','class'],as_index=False)[['score_KS','undercoverage']].mean()
rho_c,p_c=stats.spearmanr(cls_lvl['score_KS'],cls_lvl['undercoverage'])
idx=np.arange(len(cls_lvl))
bs=[]
for _ in range(B):
    pick=RNG.choice(idx,len(idx),replace=True); s=cls_lvl.iloc[pick]
    if s['score_KS'].nunique()>2:
        bs.append(stats.spearmanr(s['score_KS'],s['undercoverage'])[0])
lo,hi=np.percentile(bs,[2.5,97.5])
print(f'MECHANISM class-level: rho={rho_c:.3f} p={p_c:.2e} n={len(cls_lvl)}  95% CI [{lo:.3f}, {hi:.3f}]')
per_ds_mech={}
for ds in ['nslkdd','cicids2017','ugr16']:
    s=cls_lvl[cls_lvl['dataset'].str.startswith(ds)]
    if len(s)>3:
        r,p=stats.spearmanr(s['score_KS'],s['undercoverage']); per_ds_mech[ds]={'rho':round(float(r),3),'p':round(float(p),4),'n':len(s)}
        print(f'   within {ds:11s}: rho={r:+.3f} p={p:.4f} n={len(s)}')

# (b) monitor per dataset with AUROC CI
r_d=mo['drift_labelfree'].rank(pct=True); r_m=mo['pred_mass_drop'].clip(lower=0).rank(pct=True)
mo['lf_detector']=np.where(mo['low_support'].fillna(False),r_m,r_d)
mo['is_under']=(mo['undercoverage']>0.05).astype(int)
mon_rows=[]
for ds in ['nslkdd','cicids2017','ugr16','POOLED']:
    s=mo if ds=='POOLED' else mo[mo.ds==ds]
    rho,_=stats.spearmanr(s['lf_detector'],s['undercoverage'])
    y=s['is_under'].to_numpy(); d=s['lf_detector'].to_numpy()
    auc=roc_auc_score(y,d) if len(np.unique(y))>1 else np.nan
    ab=[]
    for _ in range(B):
        bidx=RNG.choice(len(s),len(s),replace=True)
        if len(np.unique(y[bidx]))>1: ab.append(roc_auc_score(y[bidx],d[bidx]))
    alo,ahi=(np.percentile(ab,[2.5,97.5]) if ab else (np.nan,np.nan))
    orho,_=stats.spearmanr(s['drift_truelabel'],s['undercoverage'])
    mon_rows.append({'dataset':ds,'n':len(s),'n_under':int(y.sum()),
                     'lf_rho':round(float(rho),3),'lf_auroc':round(float(auc),3),
                     'auroc_ci_lo':round(float(alo),3),'auroc_ci_hi':round(float(ahi),3),
                     'oracle_rho':round(float(orho),3)})
mon_tbl=pd.DataFrame(mon_rows)
print('\nMONITOR per dataset (authoritative; detector = pct-rank rule from nb21):')
print(mon_tbl.to_string(index=False))

# threshold sensitivity
print('\nAUROC sensitivity to the undercoverage threshold:')
for thr in [0.02,0.05,0.10,0.20]:
    y=(mo['undercoverage']>thr).astype(int)
    a=roc_auc_score(y,mo['lf_detector']) if y.nunique()>1 else np.nan
    print(f'   threshold {thr:.2f}: AUROC={a:.3f}  (n_under={int(y.sum())})')


MECHANISM class-level: rho=0.926 p=4.70e-09 n=20  95% CI [0.773, 0.986]
   within nslkdd     : rho=+0.900 p=0.0374 n=5
   within cicids2017 : rho=+0.841 p=0.0023 n=10
   within ugr16      : rho=+1.000 p=0.0000 n=5

MONITOR per dataset (authoritative; detector = pct-rank rule from nb21):
   dataset  n  n_under  lf_rho  lf_auroc  auroc_ci_lo  auroc_ci_hi  oracle_rho
    nslkdd 15       12   0.384     0.694        0.429        0.929       0.886
cicids2017 30       14   0.768     0.955        0.839        1.000       0.896
     ugr16 15        6   0.824     0.981        0.893        1.000       0.979
    POOLED 60       32   0.725     0.895        0.808        0.966       0.925

AUROC sensitivity to the undercoverage threshold:
   threshold 0.02: AUROC=0.895  (n_under=32)
   threshold 0.05: AUROC=0.895  (n_under=32)
   threshold 0.10: AUROC=0.895  (n_under=32)
   threshold 0.20: AUROC=0.888  (n_under=28)


In [12]:
# =============================================================================
# C6 - POOLED MODEL, HONEST VERSION. The committed fit clusters by unit (160) and
# returns p=1e-173 on 49,500 correlated cells, 73% of which are NSL-KDD. Refit
# clustering by DATASET (the level at which S_cov actually varies) and report how
# far the SEs widen. With 3 clusters the cluster-robust CI is itself unreliable,
# which is exactly the point: the pooled design cannot carry this claim.
# =============================================================================
import statsmodels.api as sm, statsmodels.formula.api as smf
pc=pd.read_csv(config.REPORTS_DIR/'pooled_coverage.csv')
print('pooled rows:',len(pc),'| per dataset:',pc['dataset'].value_counts().to_dict())
for c in ['S_cov','S_lab','S_sup']:
    mu,sd=pc[c].mean(),pc[c].std(); pc['z'+c.replace('S_','S')]=(pc[c]-mu)/(sd if sd else 1)
pc['protocol']=pd.Categorical(pc['protocol'],categories=['TSC','REC','SHC'])
form='n_covered + I(n_eval - n_covered) ~ C(protocol)*zScov + C(protocol)*zSlab + C(protocol)*zSsup'
res={}
for label,grp in [('unit (as published, 160 clusters)','unit_id'),('dataset (honest, 3 clusters)','dataset')]:
    try:
        m=smf.glm(form,data=pc,family=sm.families.Binomial()).fit(
            cov_type='cluster',cov_kwds={'groups':pc[grp]})
        t7='C(protocol)[T.SHC]:zSsup'; t5='C(protocol)[T.SHC]:zScov'
        res[label]={}
        for nm,t in [('beta7_SHC_x_Ssup',t7),('beta5_SHC_x_Scov',t5)]:
            if t in m.params.index:
                res[label][nm]={'beta':round(float(m.params[t]),4),'se':round(float(m.bse[t]),4),
                                'p':float(m.pvalues[t]),
                                'ci':[round(float(m.conf_int().loc[t,0]),4),round(float(m.conf_int().loc[t,1]),4)]}
        print(f'\n[{label}]')
        for nm,v in res[label].items():
            print(f'   {nm}: beta={v["beta"]:+.4f} se={v["se"]:.4f} p={v["p"]:.3g} CI={v["ci"]}')
    except Exception as e:
        print(f'\n[{label}] FIT FAILED: {e}'); res[label]={'error':str(e)}

# proposed PRIMARY: within-NSL dose-response on the ladder (SHC only)
nsl_lad=nsl_p[(nsl_p['class']=='R2L')&(np.isclose(nsl_p['alpha'],0.05))&(nsl_p['protocol']=='SHC')]
g=nsl_lad.groupby('rung')['coverage'].mean()
sl,ic,r,p,se=stats.linregress(g.index.astype(float),g.values)
print(f'\nPROPOSED PRIMARY - NSL dose-response (SHC focal vs ladder rung):')
print('  rung means:', {round(float(k),2):round(float(v),4) for k,v in g.items()})
print(f'  slope={sl:.4f} per unit rung  r={r:.3f}  p={p:.4g}  (monotone decline = dose-response)')


pooled rows: 49500 | per dataset: {'nslkdd': 36000, 'cicids2017': 9000, 'ugr16': 4500}

[unit (as published, 160 clusters)]
   beta7_SHC_x_Ssup: beta=-0.3742 se=0.0133 p=1.45e-173 CI=[-0.4004, -0.3481]
   beta5_SHC_x_Scov: beta=-0.2519 se=0.0090 p=2.33e-172 CI=[-0.2696, -0.2343]

[dataset (honest, 3 clusters)]
   beta7_SHC_x_Ssup: beta=-0.3742 se=0.0568 p=4.4e-11 CI=[-0.4856, -0.2629]
   beta5_SHC_x_Scov: beta=-0.2519 se=0.0362 p=3.54e-12 CI=[-0.3229, -0.1809]

PROPOSED PRIMARY - NSL dose-response (SHC focal vs ladder rung):
  rung means: {0.0: 0.1431, 0.2: 0.1139, 0.4: 0.0848, 0.6: 0.0582, 0.8: 0.0298}
  slope=-0.1411 per unit rung  r=-1.000  p=1.923e-06  (monotone decline = dose-response)


In [ ]:
# =============================================================================
# C7 - save everything + commit.
# =============================================================================
out={'C1_calibration_size_audit':c1,
     'C2_marginal_vs_mondrian':marg_tbl.to_dict('records'),
     'C4_inclusion_rule':{'nsl_infeasible_classes':sorted(infeas),
        'mechanism_all':{'rho':round(m_all[0],4),'n':m_all[2]},
        'mechanism_feasible_only':{'rho':round(m_fea[0],4),'n':m_fea[2]},
        'monitor_all':{'rho':round(o_all[0],4),'auc':round(o_all[1],4),'n':o_all[2]},
        'monitor_feasible_only':{'rho':round(o_fea[0],4),'auc':round(o_fea[1],4),'n':o_fea[2]}},
     'C5_mechanism_class_level':{'rho':round(float(rho_c),4),'p':float(p_c),'n':int(len(cls_lvl)),
        'ci':[round(float(lo),4),round(float(hi),4)],'per_dataset':per_ds_mech},
     'C6_pooled_clustering':res}
(config.REPORTS_DIR/'audit_corrections.json').write_text(json.dumps(out,indent=2,default=str))
ci_tbl.to_csv(config.REPORTS_DIR/'headline_bootstrap_cis.csv',index=False)
marg_tbl.to_csv(config.REPORTS_DIR/'marginal_vs_mondrian_nslkdd.csv',index=False)
mon_tbl.to_csv(config.REPORTS_DIR/'monitor_per_dataset.csv',index=False)
print('saved: audit_corrections.json, headline_bootstrap_cis.csv, marginal_vs_mondrian_nslkdd.csv, monitor_per_dataset.csv')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb27: audit corrections - calib-size check, marginal vs mondrian, bootstrap CIs, inclusion rule, per-dataset monitor, pooled clustering')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved: audit_corrections.json, headline_bootstrap_cis.csv, marginal_vs_mondrian_nslkdd.csv, monitor_per_dataset.csv
